In [1]:
# 📦 Imports
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")   # ✅ faster backend for saving images only
import matplotlib.pyplot as plt
import os
from google.colab import drive
import zipfile

# 🔗 Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ✅ Use Parquet if possible (much faster than CSV)
csv_path = "/content/drive/MyDrive/spc_data.csv"
parquet_path = "/content/drive/MyDrive/spc_data.parquet"

if not os.path.exists(parquet_path):
    df = pd.read_csv(csv_path, low_memory=False)
    df.to_parquet(parquet_path)
else:
    df = pd.read_parquet(parquet_path)

print("Data imported successfully:", df.shape)

Data imported successfully: (19036, 4200)


In [18]:
df.head()

,spc.400,spc.400.5,spc.401,spc.401.5,spc.402,spc.402.5,spc.403,spc.403.5,spc.404,spc.404.5,...,spc.2495,spc.2495.5,spc.2496,spc.2496.5,spc.2497,spc.2497.5,spc.2498,spc.2498.5,spc.2499,spc.2499.5
0,0.831705,0.839111,0.846473,0.853773,0.860988,0.868098,0.875088,0.881935,0.888626,0.895142,...,0.552283,0.552303,0.552319,0.552333,0.552342,0.552346,0.552345,0.552338,0.552329,0.552314
1,0.706027,0.714901,0.723727,0.732480,0.741142,0.749684,0.758085,0.766323,0.774371,0.782215,...,0.424424,0.424536,0.424642,0.424742,0.424837,0.424924,0.425009,0.425088,0.425165,0.425256
2,0.666238,0.676472,0.686654,0.696757,0.706753,0.716615,0.726321,0.735840,0.745154,0.754237,...,0.426412,0.426590,0.426763,0.426931,0.427094,0.427254,0.427409,0.427561,0.427712,0.427900
3,0.698136,0.706548,0.714909,0.723197,0.731384,0.739448,0.747365,0.755111,0.762668,0.770012,...,0.593780,0.594077,0.594368,0.594653,0.594935,0.595211,0.595482,0.595749,0.596013,0.596341
4,0.675433,0.684820,0.694163,0.703438,0.712620,0.721693,0.730629,0.739405,0.748003,0.756403,...,0.401095,0.401195,0.401289,0.401380,0.401467,0.401554,0.401637,0.401720,0.401801,0.401902


###  Function: `prepare_spider_data`

This function normalizes a 1D spectrum, closes the polygon loop, creates a radial grid, interpolates the spectrum on the grid, and generates a mask for values outside the polygon. These outputs are suitable for coloring/filling under the curve in a radar (spider) plot.

---

#### **Parameters**
- **`values`** : *array-like of shape (N,)*  
  The 1D spectrum values (e.g., intensity measurements for each spectral variable).

- **`r_steps`** : *int, default=200*  
  Number of radial points to create between 0 and the maximum normalized value.  
  Determines the radial resolution of the grid used for coloring.

---

#### **Returns**
- **`theta`** : *ndarray of shape (N+1,)*  
  Angular coordinates for each point in the spectrum, from 0 to 2π, with an extra point to close the polygon loop.

- **`values_norm`** : *ndarray of shape (N+1,)*  
  Normalized spectrum values scaled between 0 and 1, with an extra value appended to close the polygon.

- **`T`** : *ndarray of shape (r_steps, N+1)*  
  2D grid of angular coordinates created by `meshgrid`.

- **`R`** : *ndarray of shape (r_steps, N+1)*  
  2D grid of radial coordinates created by `meshgrid`, ranging from 0 to 1.

- **`Z`** : *ndarray of shape (r_steps, N+1)*  
  Interpolated intensity values mapped to the radial grid, with all points outside the polygon set to NaN.

- **`masked`** : *ndarray of shape (r_steps, N+1), dtype=bool*  
  Boolean mask indicating which points in the radial grid are **outside the polygon**.  
  `True = outside`, `False = inside`.

---

#### **Notes**
- Normalization ensures all spectrum values are scaled to `[0, 1]` while preserving relative peak differences.  
- The mask allows plotting libraries (e.g., `pcolormesh`) to fill **only the area under the polygon**, leaving outside points transparent.  
- This function prepares data for spider plots but does **not** perform the actual plotting.


In [3]:
# ✅ Normalize all rows in one step (row-wise normalization)
df_norm = df.div(df.max(axis=1), axis=0)

# ✅ Precompute meshgrid once (not inside loop)
N = df.shape[1]
theta = np.linspace(0, 2 * np.pi, N + 1)
r_steps = 200
r = np.linspace(0, 1, r_steps)
T, R = np.meshgrid(theta, r)

print("Meshgrid & normalization ready.")

Meshgrid & normalization ready.


In [12]:
def prepare_spider_data(values, theta, T, R):
    """Prepare Z mask for spider plot using normalized row values."""
    values = np.asarray(values, dtype=float)
    values = np.append(values, values[0])   # close polygon

    # Interpolation (fast np.interp)
    V = np.interp(T, theta, values)

    # Masking
    masked = R > V
    Z = V.copy()
    Z[masked] = np.nan
    return values, Z

def plot_spider(theta, values_norm, T, R, Z, save_path=None):
    fig, ax = plt.subplots(subplot_kw=dict(polar=True), figsize=(6,6))
    ax.pcolormesh(T, R, Z, cmap='viridis', shading='gouraud', vmin=0, vmax=1)
    ax.plot(theta, values_norm, color='black', lw=1)
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.grid(False)
    ax.set_frame_on(False)
    ax.set_ylim(0, 1)
    ax.set_facecolor('white')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight", pad_inches=0)
    else:
        plt.show()
    plt.close()


In [9]:
output_dir = "/content/Spider_images"
os.makedirs(output_dir, exist_ok=True)

batch_size = 200
total_rows = len(df_norm)

for start in range(1122, total_rows, batch_size):
    end = min(start + batch_size, total_rows)
    print(f"🔄 Processing batch {start} → {end-1}")

    for i in range(start, end):
        row = df_norm.iloc[i].to_numpy()
        values, Z = prepare_spider_data(row, theta, T, R)

        save_path = os.path.join(output_dir, f"spider_plot_{i}.png")

        # create figure with exact pixel size (65x65)
        fig = plt.figure(figsize=(0.65, 0.65), dpi=100)
        plot_spider(theta, values, T, R, Z)   # just draws
        plt.savefig(save_path, dpi=100, bbox_inches="tight", pad_inches=0)
        plt.close(fig)

    print(f"✅ Finished batch {start} → {end-1}")

print(f"All plots saved in folder: {output_dir}")


🔄 Processing batch 1122 → 1321
✅ Finished batch 1122 → 1321
🔄 Processing batch 1322 → 1521
✅ Finished batch 1322 → 1521
🔄 Processing batch 1522 → 1721
✅ Finished batch 1522 → 1721
🔄 Processing batch 1722 → 1921
✅ Finished batch 1722 → 1921
🔄 Processing batch 1922 → 2121
✅ Finished batch 1922 → 2121
🔄 Processing batch 2122 → 2321
✅ Finished batch 2122 → 2321
🔄 Processing batch 2322 → 2521
✅ Finished batch 2322 → 2521
🔄 Processing batch 2522 → 2721
✅ Finished batch 2522 → 2721
🔄 Processing batch 2722 → 2921
✅ Finished batch 2722 → 2921
🔄 Processing batch 2922 → 3121
✅ Finished batch 2922 → 3121
🔄 Processing batch 3122 → 3321
✅ Finished batch 3122 → 3321
🔄 Processing batch 3322 → 3521
✅ Finished batch 3322 → 3521
🔄 Processing batch 3522 → 3721
✅ Finished batch 3522 → 3721
🔄 Processing batch 3722 → 3921
✅ Finished batch 3722 → 3921
🔄 Processing batch 3922 → 4121
✅ Finished batch 3922 → 4121
🔄 Processing batch 4122 → 4321
✅ Finished batch 4122 → 4321
🔄 Processing batch 4322 → 4521


KeyboardInterrupt: 

In [ ]:
import os
import zipfile

output_dir = "Spider_images"
zip_filename = "Spider_images.zip"

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in os.listdir(output_dir):
        file_path = os.path.join(output_dir, file)
        zipf.write(file_path, arcname=file)

print(f"🎉 All images compressed into {zip_filename}")


In [6]:
import shutil

folder = "Spider_images"
shutil.rmtree(folder)
print(f"✅ Folder '{folder}' deleted")


✅ Folder 'Spider_images' deleted
